# Практикум 5. Аудит модели и эталонный мини-проект (Google Colab)
**После блокнота вы умеете:** (1) провести базовую валидацию модели: качество с доверительными интервалами, стабильность во времени, дрейф (PSI/KS), робастность к шуму, пропускам и шуму меток, устойчивость к обходу, справедливость по группам, стабильность объяснений, проверку утечек; (2) безопасно проверить файл модели из открытых источников (хеш, анализ опкодов pickle); (3) собрать отчёт валидации с вердиктом «допуск / допуск с ограничениями / отказ»; (4) пройти эталонный мини-проект по процессу закупок: тесты, аномалии, концентрация, программа аудита, отчёт с проверкой чисел.
**Данные:** синтетические. Модель оценки риска с группами A/B и дрейфом в «продуктиве»; журнал закупок с известными закладками.
**Артефакты:** `validation_report.md`, `p5_metrics.json`, `project_findings.csv`, `project_report_draft.md`, рисунки `fig_*.png`. Запуск: Среда выполнения → Выполнить всё.

In [ ]:
import numpy as np, pandas as pd, json, re, io, pickle, pickletools, hashlib, os
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.inspection import permutation_importance
SEED = 42; rng = np.random.default_rng(SEED); pd.options.display.float_format = '{:,.3f}'.format
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.titlesize':11})
TEAL, AMBER, RED, NAVY, GREY, BLUE = '#0E7C86', '#F2A541', '#D1495B', '#0B2545', '#9AA5B1', '#5B8DB8'
sig = lambda x: 1/(1+np.exp(-x)); style = lambda ax: (ax.grid(alpha=.25, lw=.5), ax.set_axisbelow(True))

## 1. Модель на проверке
12 000 наблюдений за 8 кварталов, группа B составляет 40%. Обучение — кварталы 1–6, валидация — 7–8, «продуктив» — кварталы 9–10 с дрейфом. Признак `proxy` коррелирует с группой (косвенный признак).

In [ ]:
def make(n, qs, drift=False):
    g = (rng.random(n) < .4).astype(int); X = pd.DataFrame(rng.normal(size=(n,6)), columns=[f'x{i}' for i in range(1,7)]); X['x1'] += .3*g
    logit = -2.2+.8*X.x1+.6*X.x2-.5*X.x3+.4*X.x4*(X.x5 > 0)+.3*X.x6+rng.normal(0,.4,n); X['proxy'] = .6*g+.5*X.x1+rng.normal(0,.5,n)
    if drift: X['x1'] += .5; X['x3'] *= 1.5
    y = (rng.random(n) < sig(logit)).astype(int); X['q'] = rng.choice(qs, n); return X, y, g
Xa, ya, ga = make(9000, list(range(1,9))); Xp, yp, gp = make(3000, [9,10], drift=True)
tr = Xa.q <= 6; FE = [c for c in Xa.columns if c != 'q']
Xtr, ytr, Xva, yva, gva = Xa[tr][FE], ya[tr], Xa[~tr][FE], ya[~tr], ga[~tr]; Xpr = Xp[FE]
gbm = HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=250, l2_regularization=1., random_state=SEED).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
s_va, s_pr = gbm.predict_proba(Xva)[:,1], gbm.predict_proba(Xpr)[:,1]
print('наблюдений: обучение', len(Xtr), '| валидация', len(Xva), '| продуктив', len(Xpr), '| доля y:', ytr.mean().round(3), yva.mean().round(3), yp.mean().round(3))

## 2. Качество с доверительными интервалами и сравнение с базовой линией

In [ ]:
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates='drop'); g = pd.DataFrame({'y':y,'p':p,'q':q}).groupby('q', observed=True); return float((g.size()/len(y)*(g.y.mean()-g.p.mean()).abs()).sum())
def boot(f, y, s, B=500, seed=1):
    r = np.random.default_rng(seed); v = [f(y[i], s[i]) for i in (r.integers(0, len(y), len(y)) for _ in range(B))]; return np.percentile(v, [2.5, 97.5])
y_ = np.asarray(yva); Q = {}
for name, s in [('GBM', s_va), ('логрегрессия', lr.predict_proba(Xva)[:,1]), ('базовая: только x1', Xva.x1.values)]:
    Q[name] = {'ROC-AUC': roc_auc_score(y_, s), 'нижняя 95%': boot(roc_auc_score, y_, s)[0], 'верхняя 95%': boot(roc_auc_score, y_, s)[1], 'PR-AUC': average_precision_score(y_, s)}
print(pd.DataFrame(Q).T); print('Brier %.3f | ECE %.3f' % (brier_score_loss(y_, s_va), ece(y_, s_va)))

## 3. Стабильность во времени и дрейф
AUC по кварталам, затем PSI/KS между обучением и продуктивом по признакам и по скору. Ориентиры PSI: <0,10 ок, 0,10–0,25 внимание, ≥0,25 дрейф.

In [ ]:
def psi(ref, cur, bins=10, eps=1e-6):
    e = np.unique(np.quantile(ref, np.linspace(0,1,bins+1))); e[0], e[-1] = -np.inf, np.inf
    p = np.clip(np.histogram(ref,e)[0]/len(ref), eps, None); q = np.clip(np.histogram(cur,e)[0]/len(cur), eps, None); return float(np.sum((p-q)*np.log(p/q)))
QA = {int(k): roc_auc_score(ya[Xa.q==k], gbm.predict_proba(Xa[Xa.q==k][FE])[:,1]) for k in range(1,9)}
for k in (9,10): QA[k] = roc_auc_score(yp[Xp.q==k], gbm.predict_proba(Xp[Xp.q==k][FE])[:,1])
print({k: round(v,3) for k,v in QA.items()})
PS = {c: psi(Xtr[c].values, Xpr[c].values) for c in FE}; PS['скор модели'] = psi(gbm.predict_proba(Xtr)[:,1], s_pr)
print({k: round(v,3) for k,v in PS.items()}); print('KS по x1: %.3f' % stats.ks_2samp(Xtr.x1, Xpr.x1).statistic)

## 4. Робастность: шум, пропуски, шум меток; обход

In [ ]:
base_auc = roc_auc_score(y_, s_va); sd = Xtr.std(); NOISE, MISS, LAB = {}, {}, {}
for s_ in [0, .1, .25, .5, 1.0]:
    r = np.random.default_rng(3); NOISE[s_] = roc_auc_score(y_, gbm.predict_proba(Xva+r.normal(0,1,Xva.shape)*sd.values*s_)[:,1])
for m in [0, .05, .1, .2, .4]:
    r = np.random.default_rng(4); Xm = Xva.mask(r.random(Xva.shape) < m); MISS[m] = roc_auc_score(y_, gbm.predict_proba(Xm)[:,1])
for f in [0, .05, .1, .2, .3]:
    r = np.random.default_rng(5); yf = np.where(r.random(len(ytr)) < f, 1-ytr, ytr)
    LAB[f] = roc_auc_score(y_, HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=250, l2_regularization=1., random_state=SEED).fit(Xtr, yf).predict_proba(Xva)[:,1])
print('шум', {k: round(v,3) for k,v in NOISE.items()}); print('пропуски', {k: round(v,3) for k,v in MISS.items()}); print('шум меток', {k: round(v,3) for k,v in LAB.items()})
thr = np.quantile(s_va, .9); EV = {}                                              # обход: нарушитель сдвигает контролируемый признак x1
for d in [0, .25, .5, 1.0, 1.5]:
    Xe = Xva.copy(); Xe.loc[y_ == 1, 'x1'] -= d*sd['x1']; EV[d] = float((gbm.predict_proba(Xe)[:,1][y_ == 1] >= thr).mean())
print('полнота при бюджете алертов 10% и сдвиге признака на δ·σ:', {k: round(v,3) for k,v in EV.items()})

## 5. Справедливость по группам, стабильность объяснений, утечки

In [ ]:
def fair(y, s, g, thr):
    a = s >= thr; sel = [a[g==k].mean() for k in (0,1)]; tpr = [a[(g==k)&(y==1)].mean() for k in (0,1)]; fpr = [a[(g==k)&(y==0)].mean() for k in (0,1)]
    return dict(sel_A=sel[0], sel_B=sel[1], DPD=sel[1]-sel[0], DIR=min(sel)/max(sel), TPR_A=tpr[0], TPR_B=tpr[1], EOD=tpr[1]-tpr[0], FPR_A=fpr[0], FPR_B=fpr[1])
gv = np.asarray(gva); F0 = fair(y_, s_va, gv, thr); r = np.random.default_rng(2); bs = [fair(y_[i], s_va[i], gv[i], thr) for i in (r.integers(0, len(y_), len(y_)) for _ in range(500))]
CI = {k: np.percentile([b[k] for b in bs], [2.5, 97.5]) for k in ('DPD','EOD','DIR')}; print({k: round(v,3) for k,v in F0.items()}); print({k: [round(x,3) for x in v] for k,v in CI.items()})
noproxy = [c for c in FE if c != 'proxy']; g2 = HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=250, l2_regularization=1., random_state=SEED).fit(Xtr[noproxy], ytr); s2 = g2.predict_proba(Xva[noproxy])[:,1]
F2 = fair(y_, s2, gv, np.quantile(s2, .9)); print('без proxy: AUC %.3f (было %.3f) | DIR %.3f | EOD %.3f' % (roc_auc_score(y_, s2), base_auc, F2['DIR'], F2['EOD']))
rk = []
for b in range(10):
    ix = np.random.default_rng(b).integers(0, len(y_), len(y_)); pi = permutation_importance(gbm, Xva.iloc[ix], y_[ix], scoring='average_precision', n_repeats=3, random_state=b); rk.append(pd.Series(pi.importances_mean, FE).rank().values)
rho = np.mean([stats.spearmanr(rk[i], rk[j])[0] for i in range(10) for j in range(i+1,10)]); print('стабильность рангов важности (Спирмен): %.3f' % rho)
single = {c: max(roc_auc_score(y_, Xva[c]), 1-roc_auc_score(y_, Xva[c])) for c in FE}; Xl = Xva.assign(leak=y_+rng.normal(0,.3,len(y_))); sl = max(roc_auc_score(y_, Xl.leak), 1-roc_auc_score(y_, Xl.leak))
print('макс. AUC одного признака %.3f | с внедрённой утечкой %.3f' % (max(single.values()), sl)); dup = pd.merge(Xtr.round(6).drop_duplicates(), Xva.round(6).drop_duplicates()).shape[0]; print('дубли обучение/валидация:', dup)

## 6. Проверка файла модели из открытых источников
Хеш фиксирует версию. Анализ опкодов pickle находит вызовы вне белого списка **без загрузки файла**. Безопасная практика: форматы без исполнения кода (safetensors), проверка хеша, изолированная среда.

In [ ]:
ALLOW = ('sklearn','numpy','scipy','collections','joblib','copyreg','_codecs','builtins')
DANGER = {'eval','exec','compile','open','__import__','getattr','system','popen'}
def scan_pickle(b):
    flags, last, memo, cnt = [], [], {}, 0
    for op, arg, pos in pickletools.genops(b):
        n = op.name; mod = nm = None
        if n in ('SHORT_BINUNICODE','BINUNICODE','UNICODE'): last.append(arg)
        elif n in ('BINGET','LONG_BINGET'): last.append(memo.get(arg))
        elif n == 'MEMOIZE': memo[cnt] = last[-1] if last else None; cnt += 1
        elif n in ('BINPUT','LONG_BINPUT'): memo[arg] = last[-1] if last else None
        elif n == 'STACK_GLOBAL' and len(last) >= 2: mod, nm = last[-2], last[-1]
        elif n in ('GLOBAL','INST'): mod, nm = str(arg).split(' ')[0], str(arg).split(' ')[-1]
        else: last.append(None)                                              # иное действие разрывает «соседство» двух строк
        if mod and (str(mod).split('.')[0] not in ALLOW or nm in DANGER): flags.append(f'{mod}.{nm}')
    return sorted(set(flags))
b_ok = pickle.dumps(lr); b_sus = pickle.dumps(os.getcwd)                          # безобидная функция: нужна только как пример вызова вне белого списка
sha = lambda b: hashlib.sha256(b).hexdigest()[:16]
SCAN = {'логрегрессия sklearn': {'sha256': sha(b_ok), 'флаги': scan_pickle(b_ok)}, 'ссылка на os-функцию': {'sha256': sha(b_sus), 'флаги': scan_pickle(b_sus)}}; print(SCAN)

## 7. Отчёт валидации и вердикт

In [ ]:
lo = Q['GBM']['нижняя 95%']; maxpsi = max(v for k, v in PS.items() if k != 'скор модели'); scorepsi = PS['скор модели']
CK = [('Нижняя граница ROC-AUC ≥ 0,70', lo, lo >= .70, True), ('ECE ≤ 0,05', ece(y_, s_va), ece(y_, s_va) <= .05, False), ('PSI скора < 0,25', scorepsi, scorepsi < .25, True), ('Макс. PSI признака < 0,10', maxpsi, maxpsi < .10, False),
      ('Потеря AUC при шуме 0,25σ ≤ 0,05', base_auc-NOISE[.25], base_auc-NOISE[.25] <= .05, False), ('Потеря AUC при 20% пропусков ≤ 0,05', base_auc-MISS[.2], base_auc-MISS[.2] <= .05, False),
      ('Потеря полноты при обходе 0,5σ ≤ 0,15', EV[0]-EV[.5], EV[0]-EV[.5] <= .15, False), ('DIR ≥ 0,80 (ориентир)', F0['DIR'], F0['DIR'] >= .8, False), ('DIR: нижняя граница 95% ≥ 0,80', CI['DIR'][0], CI['DIR'][0] >= .8, False), ('|EOD| ≤ 0,10', abs(F0['EOD']), abs(F0['EOD']) <= .10, False),
      ('Стабильность важности ≥ 0,80', rho, rho >= .8, False), ('Нет утечки (AUC признака < 0,95)', max(single.values()), max(single.values()) < .95, True), ('Файл модели без подозрительных вызовов', len(SCAN['логрегрессия sklearn']['флаги']), len(SCAN['логрегрессия sklearn']['флаги']) == 0, True)]
V = pd.DataFrame(CK, columns=['проверка','значение','пройдена','блокер']); V['статус'] = np.where(V.пройдена, 'ок', np.where(V.блокер, 'БЛОКЕР', 'ВНИМАНИЕ'))
verdict = 'ОТКАЗ' if (~V.пройдена & V.блокер).any() else ('ДОПУСК С ОГРАНИЧЕНИЯМИ' if (~V.пройдена).any() else 'ДОПУСК'); print(V[['проверка','значение','статус']].to_string()); print('ВЕРДИКТ:', verdict)
open('validation_report.md','w').write('# Отчёт валидации\n\nВердикт: **'+verdict+'**\n\n```\n'+V[['проверка','значение','статус']].to_string()+'\n```')

## 8. Эталонный мини-проект: процесс закупок
Шаги проекта: карта процесса и рисков → тесты → аномалии → концентрация → программа аудита → отчёт с проверкой чисел. Закладки известны, поэтому можно оценить точность каждого теста.

In [ ]:
N, T_ = 8000, 100_000; CATS = ['Канцелярия','ИТ','Услуги','Оборудование','Консалтинг']; MED = dict(zip(CATS, [20e3, 40e3, 15e3, 60e3, 30e3]))
sup = pd.DataFrame({'sid': [f'P{i:03d}' for i in range(220)], 'cat': rng.choice(CATS, 220)}); P = pd.DataFrame({'doc': np.arange(N)}); P['cat'] = rng.choice(CATS, N); P['req'] = rng.integers(1,13,N)
P['sid'] = [rng.choice(sup[sup.cat == c].sid) if rng.random() > (.55 if c == 'ИТ' else 0) else sup[sup.cat == 'ИТ'].sid.iloc[0] for c in P.cat]; P['cat'] = P.sid.map(sup.set_index('sid').cat)
P['amount'] = np.round(np.minimum(rng.lognormal(np.log(P.cat.map(MED)), .8), 95e3), 2); P['appr'] = (P.req + rng.integers(1,6,N)) % 12 + 1
WD = pd.bdate_range('2026-01-01', periods=130); P['date'] = pd.to_datetime(rng.choice(WD, N)) + pd.to_timedelta(rng.integers(9*3600, 18*3600, N), 's'); P['inv'] = [f'N{rng.integers(10**6,10**7)}' for _ in range(N)]; P['label'] = ''
def mark(ix, lab): P.loc[ix, 'label'] = lab
free = list(rng.permutation(N)); take = lambda k: [free.pop() for _ in range(k)]
for _ in range(60):                                                   # дробление: 3 документа одного поставщика за 1–3 дня
    ix = take(3); s0 = rng.choice(sup.sid); d0 = pd.Timestamp('2026-01-01') + pd.Timedelta(days=int(rng.integers(0,175))); P.loc[ix, 'sid'] = s0; P.loc[ix, 'cat'] = sup.set_index('sid').cat[s0]
    P.loc[ix, 'amount'] = np.round(rng.uniform(70e3, 98e3, 3), 2); P.loc[ix, 'req'] = rng.integers(1,13); P.loc[ix, 'date'] = pd.Timestamp(rng.choice(WD[:120])) + pd.to_timedelta(rng.integers(9*3600, 18*3600, 3) + rng.integers(0, 3, 3)*86400, 's'); mark(ix, 'дробление')
for i in take(60): j = i; P.loc[j, ['sid','amount','inv']] = P.loc[j-1 if j > 0 else 1, ['sid','amount','inv']].values; mark([j], 'дубль')
ix = take(80); P.loc[ix, 'appr'] = P.loc[ix, 'req']; mark(ix, 'самосогласование')
ix = take(100); P.loc[ix, 'date'] = [d.normalize() + pd.Timedelta(days=int((5-d.dayofweek) % 7), hours=int(rng.choice([1,2,3,14]))) for d in P.loc[ix, 'date']]; mark(ix, 'вне графика')
ix = take(50); P.loc[ix, 'amount'] = np.round(P.loc[ix, 'cat'].map(MED).values*rng.uniform(8,12,50), 2); mark(ix, 'цена'); P['hour'] = P.date.dt.hour; P['wk'] = P.date.dt.dayofweek >= 5
print(P.label.value_counts().to_dict(), '| документов', len(P))

In [ ]:
def t_split(P):
    r = set(); s = P[(P.amount < T_)].sort_values(['sid','date'])
    for sid, g in s.groupby(['sid','req']):
        d = g.date.values; a = g.amount.values; idx = g.doc.values
        for i in range(len(g)):
            m = (d >= d[i]) & (d <= d[i] + np.timedelta64(3,'D'))
            if m.sum() >= 2 and a[m].sum() > T_ and a[m].min() > .65*T_: r |= set(idx[m])
    return r
def t_dup(P): k = P.duplicated(['sid','amount','inv'], keep='first'); return set(P[k].doc)
def t_self(P): return set(P[P.req == P.appr].doc)
def t_off(P): return set(P[(P.wk) | (P.hour < 6)].doc)
def t_price(P): la = np.log(P.amount); z = la.groupby(P.cat).transform(lambda x: 0.6745*(x-x.median())/(x-x.median()).abs().median()); return set(P[z > 2.5].doc)
TESTS = {'дробление': t_split, 'дубль': t_dup, 'самосогласование': t_self, 'вне графика': t_off, 'цена': t_price}; RES = {}
for lab, f in TESTS.items():
    fl = f(P); truth = set(P[P.label == lab].doc); tp = len(fl & truth); RES[lab] = {'сработало': len(fl), 'precision': tp/max(len(fl),1), 'полнота': tp/len(truth)}
print(pd.DataFrame(RES).T)
sh = P.groupby(['cat','sid']).amount.sum().groupby(level=0).apply(lambda s: s/s.sum()); HH = sh.groupby(level=0).apply(lambda s: float((s**2).sum()*10000)); TOP = sh.groupby(level=0).max(); print(pd.DataFrame({'HHI': HH, 'доля топ-поставщика': TOP}))

In [ ]:
F = pd.DataFrame({'la': np.log1p(P.amount), 'near': P.amount.between(.7*T_, T_).astype(int), 'hour': P.hour, 'wk': P.wk.astype(int), 'self': (P.req == P.appr).astype(int), 'dup': P.duplicated(['sid','amount','inv'], keep='first').astype(int), 'ssh': P.sid.map(P.groupby('sid').amount.sum()/P.amount.sum())})
s = -IsolationForest(n_estimators=300, random_state=SEED).fit(F).score_samples(F); y = (P.label != '').values; top = np.argsort(-s)[:200]
print('Isolation Forest: P@200 = %.3f, доля закладок %.3f' % (y[top].mean(), y.mean()))
FIND = P.assign(score=s).sort_values('score', ascending=False).head(20)[['doc','sid','cat','amount','date','label','score']]; FIND.to_csv('project_findings.csv', index=False); print(FIND.head(5).to_string())
SRC = {'S1': ('Документов закупок', len(P)), 'S2': ('Срабатываний теста дробления', RES['дробление']['сработало']), 'S3': ('Срабатываний теста самосогласования', RES['самосогласование']['сработало']), 'S4': ('Доля топ-поставщика в ИТ', round(100*TOP['ИТ'],1))}
CL = [f"Проанализировано {SRC['S1'][1]} документов закупок [S1].", f"Тест дробления сработал на {SRC['S2'][1]} документах [S2].", f"Самосогласование выявлено в {SRC['S3'][1]} документах [S3].", f"Доля топ-поставщика в категории ИТ составила {SRC['S4'][1]}% [S4]."]
ok = [all(str(v) in c.replace(' ','') or f'{v}' in c for _, v in [SRC[re.search(r'S\d', c).group()]]) for c in CL]; print('проверка чисел отчёта:', ok)
open('project_report_draft.md','w').write('# Черновик отчёта: процесс закупок\n\n'+'\n'.join('- '+c for c in CL)+'\n\nМетод: правила + Isolation Forest; ограничения: синтетические данные, закладки известны.')

## 9. Рисунки для презентации

In [ ]:
# 1. PSI
fig, ax = plt.subplots(figsize=(5.6,3.9)); k_ = list(PS); v_ = [PS[k] for k in k_]; ax.barh(k_, v_, color=[RED if v >= .25 else AMBER if v >= .1 else TEAL for v in v_]); ax.axvline(.1, c=AMBER, ls='--'); ax.axvline(.25, c=RED, ls='--'); ax.set_xlabel('PSI: обучение → продуктив'); ax.set_title('Дрейф признаков и скора'); style(ax); fig.tight_layout(); fig.savefig('fig_psi.png', dpi=200)
# 2. Стабильность
fig, ax = plt.subplots(figsize=(5.6,3.9)); ks = list(QA); ax.plot(ks, [QA[k] for k in ks], 'o-', c=TEAL, lw=2); ax.axvspan(6.5, 8.5, color=AMBER, alpha=.15, label='валидация'); ax.axvspan(8.5, 10.5, color=RED, alpha=.12, label='продуктив (дрейф)'); ax.set_ylim(.6, .85)
ax.set_xlabel('квартал'); ax.set_ylabel('ROC-AUC'); ax.set_title('Качество по кварталам'); ax.legend(frameon=False, loc='lower left'); style(ax); fig.tight_layout(); fig.savefig('fig_stability.png', dpi=200)
# 3. Робастность
fig, ax = plt.subplots(figsize=(5.6,3.9)); ax.plot(list(NOISE), list(NOISE.values()), 'o-', c=TEAL, lw=2, label='шум в признаках (доля σ)'); ax.plot(list(MISS), list(MISS.values()), 's-', c=AMBER, lw=2, label='доля пропусков'); ax.plot(list(LAB), list(LAB.values()), '^-', c=RED, lw=2, label='доля перевёрнутых меток')
ax.set_xlabel('уровень возмущения'); ax.set_ylabel('ROC-AUC'); ax.set_title('Робастность модели'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_robust.png', dpi=200)
# 4. Обход
fig, ax = plt.subplots(figsize=(5.6,3.9)); ax.plot(list(EV), list(EV.values()), 'o-', c=RED, lw=2); ax.set_xlabel('сдвиг контролируемого признака, σ'); ax.set_ylabel('полнота при бюджете алертов 10%'); ax.set_ylim(0, 1); ax.set_title('Обход детектора'); style(ax); fig.tight_layout(); fig.savefig('fig_evasion.png', dpi=200)
# 5. Справедливость
fig, ax = plt.subplots(figsize=(5.6,3.9)); lab = ['доля отбора', 'TPR', 'FPR']; A_ = [F0['sel_A'], F0['TPR_A'], F0['FPR_A']]; B_ = [F0['sel_B'], F0['TPR_B'], F0['FPR_B']]; x = np.arange(3)
ax.bar(x-.2, A_, .4, color=TEAL, label='группа A'); ax.bar(x+.2, B_, .4, color=AMBER, label='группа B'); ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_title('Метрики по группам (порог: топ-10%)'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_fairness.png', dpi=200)
# 6. Дробление
fig, ax = plt.subplots(figsize=(4.4,3.4)); ax.hist(P.amount.clip(upper=130e3)/1e3, bins=np.arange(40,131,3), color=TEAL, alpha=.85); ax.axvline(T_/1e3, c=RED, ls='--'); ax.text(T_/1e3+1, ax.get_ylim()[1]*.85, 'порог 100 тыс.', color=RED, fontsize=9)
ax.set_xlabel('сумма закупки, тыс.'); ax.set_ylabel('документов'); ax.set_title('Всплеск под порогом'); style(ax); fig.tight_layout(); fig.savefig('fig_split.png', dpi=200)
# 7. Концентрация
fig, ax = plt.subplots(figsize=(4.4,3.4)); ax.bar(TOP.index, TOP.values*100, color=[RED if v > .4 else TEAL for v in TOP.values]); ax.axhline(40, c=AMBER, ls='--'); ax.set_ylabel('доля топ-поставщика, %'); ax.set_title('Концентрация поставщиков'); plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8); style(ax); fig.tight_layout(); fig.savefig('fig_hhi.png', dpi=200)

In [ ]:
json.dump({'quality': Q, 'brier': brier_score_loss(y_, s_va), 'ece': ece(y_, s_va), 'auc_by_quarter': QA, 'psi': PS, 'noise': NOISE, 'missing': MISS, 'label_noise': LAB, 'evasion': EV, 'fairness': F0, 'fairness_ci': {k: [float(x) for x in v] for k, v in CI.items()}, 'no_proxy': {'auc': float(roc_auc_score(y_, s2)), 'DIR': F2['DIR'], 'EOD': F2['EOD']},
           'importance_rank_corr': float(rho), 'max_single_auc': max(single.values()), 'leak_injected_auc': sl, 'artifact_scan': SCAN, 'verdict': verdict, 'checks': V[['проверка','значение','статус']].to_dict('records'), 'project': {'tests': RES, 'HHI': HH.to_dict(), 'top_share': TOP.to_dict(), 'iforest_P200': float(y[top].mean()), 'base_rate': float(y.mean())}},
          open('p5_metrics.json','w'), indent=1, ensure_ascii=False, default=float)

## Задания
1. Измените порог отбора для справедливости (топ-10% → топ-5%) и повторите блок 5: как меняются DIR и EOD?
2. Добавьте проверку калибровки по группам (ECE для A и B) в отчёт валидации.
3. Добавьте в `scan_pickle` свой список запрещённых имён и проверьте на трёх файлах.
4. В проекте закупок добавьте тест «поставщик без истории + крупная сумма» и измерите precision.
5. Постройте свой отчёт по шаблону: 5 утверждений, у каждого ссылка на источник, числа проверены кодом.